## Label preparation (B2.4)
For each 8×8 patch, converts the bundled ice chart's per-pixel total-concentration (CT) data into per-patch label statistics: a per-class area-fraction vector (`frac_sic0`...`frac_sic10`), a label-coverage fraction (`valid_class_fraction`), a discrete class `label`, and a class-based `is_pure` flag.

**Bug found in B2.1's existing `chart_ct` field**: `training.data_loader.labels.build_chart_ct` divides every raw CT code by 10 without excluding the SIGRID-3 "unknown / not-filled / glacier" special codes (91, 92 — both present in this scene's `polygon_codes`). That silently produces invalid non-integer classes (9.1, 9.2) instead of excluding those pixels. Per the instruction not to touch `src/training` yet, this notebook rebuilds a *corrected* per-pixel class map locally from the raw polygon data rather than reusing `chip.chart_ct` — `compute_patch_labels` itself stays here, not promoted to a `training` module.

In [ ]:
from pathlib import Path

import numpy as np
import xarray as xr

from training import ALL_BANDS, CHIP_SIZE, GRID_SIZE, PATCH_SIZE, load_band_means, load_scene, yield_chips

### Configuration

In [ ]:
BUCKET = "prescient-ice-data"
STATS_KEY = "training_data/ai4arctic/statistics/dataset_stats.json"
AWS_PROFILE = "spk_data"

SCENE_PATH = Path(
    "../../S1A_EW_GRDM_1SDH_20180124T194759_20180124T194859_020301_022AA4_1F75"
    "_icechart_dmi_201801241950_SouthEast_RIC.nc"
)

### Load one scene and get a single chip (B2.1)
Only used here for the patch grid geometry (`chip_row_start`/`chip_col_start`) and, for comparison, the SAR `valid_mask` — label prep doesn't need the band-mean substitution at all, since it operates purely on the chart CT data.

In [ ]:
band_means = load_band_means(BUCKET, STATS_KEY, ALL_BANDS, profile=AWS_PROFILE)
scene = load_scene(SCENE_PATH, band_means)

# Chip 0 (top-left) has no ice-chart polygon coverage at all for this scene --
# chart coverage and SAR validity are independent. Chip 943 has real chart
# data spanning two classes, including at least one mixed/boundary patch.
chips = yield_chips(scene)
for _ in range(944):
    chip = next(chips)

print(f"chip_id: {chip.chip_id}")
print(f"SAR valid_fraction: {chip.valid_mask.mean():.3f}")

### Resolve per-pixel CT codes to the 0–10 class scale (SIGRID-3)
Range codes (e.g. `'50-70'`) resolve to their midpoint. Genuine single CT codes are always exact multiples of 10 (0, 10, ..., 100); any single code that isn't (91, 92 in this scene) is a SIGRID-3 unknown/not-filled/glacier sentinel and is excluded, alongside the `-9` fill code. This generalises to whatever special codes appear, rather than hardcoding 91/92 specifically.

In [ ]:
def resolve_ct_to_class(ct_str):
    """
    Resolve one raw SIGRID-3 CT code to a class 0-10, or None if the pixel
    carries no valid class (fill/no-data, or an unknown/not-filled/glacier
    special code).
    """
    s = ct_str.strip()
    if s == "-9":
        return None
    if "-" in s[1:]:  # range code, e.g. '50-70' -> midpoint 60 -> class 6
        lo, _, hi = s.partition("-")
        midpoint = (float(lo) + float(hi)) / 2.0
        return int(round(midpoint / 10.0))
    value = float(s)
    if value % 10 != 0:
        return None  # SIGRID-3 special code (unknown / not-filled / glacier)
    return int(value / 10.0)

### Build the corrected per-pixel class map for the chip
Re-reads the raw `polygon_icechart`/`polygon_codes` (the same source B2.1 reads), builds a polygon-id → class lookup using the corrected resolver above, and maps it onto the chip's pixel window. `-1` marks pixels with no valid class.

In [ ]:
with xr.open_dataset(SCENE_PATH, engine="netcdf4") as ds:
    poly_chart = ds["polygon_icechart"].values
    poly_codes = ds["polygon_codes"].values

header = str(poly_codes[0]).split(";")
ct_col = header.index("CT")

class_lookup = {}
for row in poly_codes[1:]:
    parts = str(row).split(";")
    resolved = resolve_ct_to_class(parts[ct_col])
    if resolved is not None:
        class_lookup[int(parts[0])] = resolved

if np.issubdtype(poly_chart.dtype, np.floating):
    valid_polygon = ~np.isnan(poly_chart)
    poly_ids_int = np.where(valid_polygon, poly_chart.astype(np.int32), 0)
else:
    valid_polygon = poly_chart != 65535
    poly_ids_int = poly_chart.astype(np.int32)

class_map_full = np.full(poly_chart.shape, -1, dtype=np.int8)
if valid_polygon.any():
    max_pid = int(poly_ids_int[valid_polygon].max())
    class_vec = np.full(max_pid + 1, -1, dtype=np.int8)
    for pid, cls in class_lookup.items():
        if pid <= max_pid:
            class_vec[pid] = cls
    class_map_full[valid_polygon] = class_vec[poly_ids_int[valid_polygon]]

r0, c0 = chip.chip_row_start, chip.chip_col_start
chip_class_map = class_map_full[r0 : r0 + CHIP_SIZE, c0 : c0 + CHIP_SIZE]

print(f"Valid-class pixels in chip: {(chip_class_map >= 0).sum()} / {chip_class_map.size}")
print(f"Classes present: {sorted(set(chip_class_map[chip_class_map >= 0].tolist()))}")

### Patch-level label aggregation
For each 8×8 patch: histogram the valid-class pixels into `frac_sic0`...`frac_sic10` (normalised over valid-class pixels only, so they sum to 1.0); `valid_class_fraction` is valid-class pixels / 64 (distinct from the SAR `valid_fraction` from B2.1); `label` is the area-weighted collapse (Σ class × fraction, rounded) — pure cells are the degenerate case of the same formula, not a separate code path; `is_pure` is class-based (one class at fraction 1.0). Patches with zero valid-class pixels get `label = NaN` and `is_pure = False` — there's no evidence to assign a class from.

In [ ]:
def compute_patch_labels(chip_class_map, chip_id):
    """
    Given a chip's per-pixel class map (256, 256) int8 with -1 = no valid
    class, compute per-patch label statistics for all 1024 patches in the
    32x32 grid. Returns a list of 1024 dicts, one per patch, in row-major
    order.
    """
    n_classes = 11  # 0..10 tenths
    label_records = []

    for pi in range(GRID_SIZE):
        for pj in range(GRID_SIZE):
            r0, r1 = pi * PATCH_SIZE, (pi + 1) * PATCH_SIZE
            c0, c1 = pj * PATCH_SIZE, (pj + 1) * PATCH_SIZE

            patch_classes = chip_class_map[r0:r1, c0:c1]  # (8, 8) int8
            valid_class = patch_classes >= 0
            n_valid = int(valid_class.sum())
            valid_class_fraction = n_valid / 64.0

            if n_valid > 0:
                counts = np.bincount(patch_classes[valid_class], minlength=n_classes)[:n_classes]
                fractions = counts / n_valid
                label = float(round(float(np.dot(np.arange(n_classes), fractions))))
                is_pure = bool((fractions == 1.0).any())
            else:
                fractions = np.full(n_classes, np.nan)
                label = float("nan")
                is_pure = False

            record = {
                "chip_id": chip_id,
                "patch_i": pi,
                "patch_j": pj,
                "valid_class_fraction": valid_class_fraction,
                "label": label,
                "is_pure": is_pure,
                **{f"frac_sic{k}": float(fractions[k]) for k in range(n_classes)},
            }
            label_records.append(record)

    return label_records

### Compute labels for the chip

In [ ]:
labels = compute_patch_labels(chip_class_map, chip.chip_id)
print(f"Patches: {len(labels)}")  # should be 1024 (32 x 32)
print(labels[0])

### Verify fraction vectors sum to 1.0
For every patch with at least one valid-class pixel, `frac_sic0`...`frac_sic10` should sum to 1.0 within floating-point tolerance.

In [ ]:
labelled = [r for r in labels if r["valid_class_fraction"] > 0]
sums = [sum(r[f"frac_sic{k}"] for k in range(11)) for r in labelled]

print(f"Patches with valid-class pixels: {len(labelled)} / {len(labels)}")
print(f"All fraction vectors sum to 1.0: {np.allclose(sums, 1.0)}")

### Spot-check a hand-counted pure patch and a hand-counted boundary patch
Finds one pure patch (all valid pixels one class) and one mixed/boundary patch (valid pixels span more than one class), then manually recomputes each directly from `chip_class_map` to confirm `compute_patch_labels` agrees.

In [ ]:
pure_examples = [r for r in labelled if r["is_pure"]]
mixed_examples = [r for r in labelled if not r["is_pure"]]
print(f"Pure patches: {len(pure_examples)}   Mixed/boundary patches: {len(mixed_examples)}")


def manual_check(record):
    pi, pj = record["patch_i"], record["patch_j"]
    r0, r1 = pi * PATCH_SIZE, (pi + 1) * PATCH_SIZE
    c0, c1 = pj * PATCH_SIZE, (pj + 1) * PATCH_SIZE
    patch = chip_class_map[r0:r1, c0:c1]
    valid = patch[patch >= 0]
    vals, counts = np.unique(valid, return_counts=True)
    manual_fracs = dict(zip(vals.tolist(), (counts / len(valid)).tolist()))
    manual_label = round(sum(k * f for k, f in manual_fracs.items()))
    print(f"  patch [{pi},{pj}]: hand-counted fractions = {manual_fracs}")
    print(
        f"  hand label={manual_label} vs module label={record['label']:.0f}, "
        f"is_pure={record['is_pure']}"
    )


if pure_examples:
    print("\nPure patch:")
    manual_check(pure_examples[0])

if mixed_examples:
    print("\nBoundary patch:")
    manual_check(mixed_examples[0])